In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
)
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)
from sklearn.pipeline import Pipeline
import json

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [ ]:
ct = ColumnTransformer(
    [
        (
            "Categorical columns",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include="str"),
        ),
        (
            "Age",
            KBinsDiscretizer(n_bins=5, encode="onehot", strategy="quantile"),
            ["age"],
        ),
        ("Balance", PowerTransformer(method="yeo-johnson"), ["balance"]),
    ],
    remainder="passthrough",
)

log_reg_pipe = Pipeline(
    [("preprocessor", ct), ("model", LogisticRegression(max_iter=1000))]
)

with open("../models/threshold.json", "r") as f:
    data = json.load(f)

threshold = data["threshold"]
print(threshold)

In [ ]:
log_reg_pipe.fit(X_train, y_train)
proba = log_reg_pipe.predict_proba(X_test)[:, 1]
print(proba)

In [ ]:
prec, rec, thr = precision_recall_curve(y_test, proba)
ap = average_precision_score(y_test, proba)
plt.plot(rec, prec, label=f"Average precision score: {ap:.3f}")
plt.axhline(0.117, ls="--", color="gray", label="base rate = 0.117")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.title("PR-AUC Curve")
plt.savefig("../reports/figures/test-PR-AUC.png")
plt.show()

In [ ]:
fpr, tpr, thr = roc_curve(y_test, proba)
roc_auc = roc_auc_score(y_test, proba)
plt.plot(fpr, tpr, label=f"Model ROC-AUC score: {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], label="no-skill: 0.5", ls="--")
plt.ylabel("True Positive Rate")
plt.xlabel("False Positive Rate")
plt.legend()
plt.title("ROC-AUC Curve")
plt.savefig("../reports/figures/test-ROC-AUC.png")
plt.show()

In [ ]:
y_pred = (proba >= threshold).astype(int)
print(y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
tp = ((y_pred == 1) & (y_test == 1)).sum()
fp = ((y_pred == 1) & (y_test == 0)).sum()
calls = tp + fp
money = tp * 100 - calls * 10
print(money)

a threshold tuned on out-of-fold data transferred to the untouched test at 0.69 recall / 0.24 precision against 0.67 / 0.24
  predicted, earning €4.75/lead against a €4.61 forecast.

In [ ]:
joblib.dump(log_reg_pipe, "../models/model.joblib")